# 🧬 Mikrobioma Usus Bayi: Perbandingan Persalinan Vaginal vs Sesar dengan 16S Long-Read

**Modul hands-on Google Colab | OmicsLite**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmicsLite/Workshop-Long-Reads-Metagenomics/blob/main/hands-on/gut_microbiome_16s.ipynb)

> **Istilah penting:** praktikum ini menganalisis **amplicon gen 16S rRNA**, bukan *shotgun metagenomics*. Amplicon 16S menargetkan satu gen penanda untuk membuat profil taksonomi bakteri/arkaea; ia tidak mengukur seluruh gen di komunitas dan tidak dapat langsung menyimpulkan fungsi metabolik.

## Tujuan belajar

Setelah menyelesaikan notebook ini, peserta mampu:

1. menjelaskan isi FASTQ dan skor kualitas Phred;
2. menilai distribusi panjang dan kualitas read ONT;
3. menyaring read 16S long-read (V3–V9) secara reproducible;
4. mengklasifikasikan read dengan Kraken2 + NCBI Targeted Loci (16S/18S);
5. membuat tabel komposisi dan visualisasi Krona;
6. membandingkan alpha/beta diversity antara kelompok persalinan;
7. menjalankan uji kelompok eksploratif dan menafsirkan keterbatasannya.

## Mengapa 16S long-read?

Gen 16S rRNA (~1.500 bp) memiliki daerah lestari universal untuk primer PCR dan sembilan daerah variabel (V1–V9). Dataset praktikum ini (PRJNA637202; Angell et al., 2020) menargetkan wilayah V3–V9 (~1.140 bp) yang informatif secara taksonomi. ONT dapat membaca amplicon panjang, bahkan mendekati seluruh gen. Namun, read dataset ini berpusat sekitar 1,14 kb sehingga tidak semuanya full-length utuh. Resolusi umumnya lebih baik daripada hanya satu region pendek, tetapi **identifikasi spesies tidak otomatis akurat**: kualitas read, primer, database, kemiripan antarspesies, dan variasi jumlah kopi gen 16S tetap membatasi interpretasi.

## Alur praktikum

```text
ENA FASTQ → validasi → NanoPlot → Chopper (atau SeqKit) → Kraken2/NCBI-Targeted-Loci
                                           ├→ tabel komposisi
                                           ├→ Krona
                                           └→ rarefaction → alpha & beta diversity
```

**Estimasi:** unduhan 12 FASTQ ±510 MB, database ±8.6 MB, dan waktu total sekitar 25–45 menit pada Colab CPU (bergantung jaringan/solver). Jalankan cell berurutan dari atas. Runtime Colab bersifat sementara; unduh hasil sebelum sesi berakhir.

---
## 0. Konfigurasi praktikum

Semua parameter penting dikumpulkan di satu tempat. Untuk workshop, jangan mengubahnya sebelum pipeline berhasil sekali. `SEED` memastikan rarefaction dapat direproduksi.

In [ ]:
# @title 0.1 Parameter analisis & struktur folder terorganisir
from pathlib import Path

WORKDIR = Path("/content/16s_workshop") if Path("/content").exists() else Path.cwd() / "16s_workshop"
RAW_DIR = WORKDIR / "raw"
FILTERED_DIR = WORKDIR / "filtered"
QC_DIR = WORKDIR / "qc"
DB_DIR = WORKDIR / "database_16s"

# Struktur folder hasil terorganisir per tahap analisis (traceability & reproducibility)
RESULTS_DIR = WORKDIR / "results"
RES_QC_DIR = RESULTS_DIR / "01_qc"
RES_CLASS_DIR = RESULTS_DIR / "02_classification"
RES_CLASS_REPORTS_DIR = RES_CLASS_DIR / "reports"
RES_CLASS_PER_READ_DIR = RES_CLASS_DIR / "per_read"
RES_ABUND_TABLES_DIR = RESULTS_DIR / "03_abundance" / "tables"
RES_ABUND_FIGS_DIR = RESULTS_DIR / "03_abundance" / "figures"
RES_KRONA_HTML_DIR = RESULTS_DIR / "04_krona" / "html"
RES_KRONA_TSV_DIR = RESULTS_DIR / "04_krona" / "tsv"
RES_ALPHA_DIR = RESULTS_DIR / "05_diversity" / "alpha"
RES_BETA_DIR = RESULTS_DIR / "05_diversity" / "beta"
RES_DIFF_DIR = RESULTS_DIR / "06_differential_abundance"

MIN_LENGTH = 1000
MAX_LENGTH = 1700
MIN_MEAN_Q = 10
KRAKEN_CONFIDENCE = 0.01
THREADS = 2
SEED = 42

ALL_DIRS = (
    RAW_DIR, FILTERED_DIR, QC_DIR, DB_DIR,
    RES_QC_DIR, RES_CLASS_REPORTS_DIR, RES_CLASS_PER_READ_DIR,
    RES_ABUND_TABLES_DIR, RES_ABUND_FIGS_DIR,
    RES_KRONA_HTML_DIR, RES_KRONA_TSV_DIR,
    RES_ALPHA_DIR, RES_BETA_DIR, RES_DIFF_DIR,
)
for directory in ALL_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Workspace : {WORKDIR}")
print(f"Hasil     : {RESULTS_DIR} (terbagi ke subfolder 01_qc s.d. 06_differential_abundance)")
print(f"Filter    : {MIN_LENGTH}–{MAX_LENGTH} bp; mean Q ≥ {MIN_MEAN_Q}")


---
## 1. Instalasi alat bioinformatika

Notebook memakai **micromamba** agar alat command-line terisolasi dari Python bawaan Colab dan runtime tidak perlu restart. Versi alat ditampilkan setelah instalasi untuk jejak reproducibility.

In [ ]:
# @title 1.1 Instal Chopper, Kraken2, Krona, NanoPlot, dan SeqKit
import os
import shutil
import subprocess
from pathlib import Path

MICROMAMBA = (Path("/content") if Path("/content").exists() else WORKDIR) / "micromamba"
ENV_PREFIX = Path("/content/bioenv") if Path("/content").exists() else WORKDIR / "bioenv"
REQUIRED_TOOLS = ("chopper", "kraken2", "ktImportText", "NanoPlot", "seqkit")

def run(cmd, *, stdout=None, cwd=None):
    """Jalankan command dan hentikan notebook segera bila command gagal."""
    printable = " ".join(map(str, cmd))
    print(f"$ {printable}")
    return subprocess.run(list(map(str, cmd)), check=True, stdout=stdout, cwd=cwd)

if not all(shutil.which(cmd) for cmd in REQUIRED_TOOLS):
    if not MICROMAMBA.exists():
        run(["bash", "-lc", "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba"])
        shutil.copy2("/tmp/bin/micromamba", MICROMAMBA)
        MICROMAMBA.chmod(0o755)

    if not (ENV_PREFIX / "bin" / "kraken2").exists():
        run([
            MICROMAMBA, "create", "-y", "-p", ENV_PREFIX,
            "--override-channels", "-c", "conda-forge", "-c", "bioconda",
            "--strict-channel-priority",
            "python=3.11", "chopper>=0.8", "kraken2>=2.1", "krona>=2.8",
            "nanoplot>=1.40", "seqkit>=2.8",
        ])

    os.environ["PATH"] = f"{ENV_PREFIX / 'bin'}:{os.environ['PATH']}"
else:
    print("ℹ️ Semua alat bioinformatika sudah tersedia di PATH lokal; instalasi micromamba dilewati.")

for command in REQUIRED_TOOLS:
    assert shutil.which(command), f"{command} tidak ditemukan di PATH"

run(["kraken2", "--version"])
run(["chopper", "--version"])
run(["seqkit", "version"])
print("\n✅ Semua alat siap.")


### Mengapa fokus klasifikasi utama menggunakan Kraken2 dan apa peran model Bracken?

Database resmi ONT EPI2ME `ncbi_16s_18s` menyediakan file distribusi k-mer `database1000mers.kmer_distrib` (~1.000 bp), yang sangat cocok dengan panjang amplikon kita (~1.140 bp). 

Namun, untuk alur praktikum pembelajaran ini, kita mengandalkan **clade count langsung dari laporan hierarki Kraken2**. Mengapa?
1. **Transparansi Taksonomi**: Peserta dapat melihat secara langsung bagaimana *reads* dialokasikan ke tingkat Phylum, Family, Genus, dan Species berdasarkan bukti k-mer nyata tanpa proses redistribusi probabilistik "kotak hitam".
2. **Kesesuaian dengan Profil Long-Read**: Kraken2 dengan ambang batas `--confidence 0.01` memberikan penugasan yang jujur—reads yang ambigu ditempatkan pada Lowest Common Ancestor (LCA) di tingkat Genus atau Family, sedangkan reads yang spesifik ditugaskan ke tingkat Species.

Bagi peserta yang membutuhkan estimasi kelimpahan spesies tervalidasi penuh pada riset mandiri, alat mutakhir berbasis *Expectation-Maximization* (**Emu**) atau pengujian model Bracken 1.000 bp dapat dievaluasi lebih lanjut.

In [ ]:
# @title 1.2 Unduh dan validasi database NCBI Targeted Loci 16S/18S (ONT EPI2ME)
import hashlib

DB_URL = "https://s3.eu-west-1.amazonaws.com/ont-open-data/workflow-databases/wf-metagenomics-dbs/ncbi_16s_18s/ncbi_targeted_loci_kraken2.tar.gz"
DB_ARCHIVE = WORKDIR / "ncbi_targeted_loci_kraken2.tar.gz"
EXPECTED_SHA256 = "f11135cdafebc20d179032b50a72b1ccc5a60b959f76b3f1b6b3198e67ca6aae"

required_db_files = [DB_DIR / name for name in ("hash.k2d", "opts.k2d", "taxo.k2d")]
if not all(path.exists() for path in required_db_files):
    if not DB_ARCHIVE.exists() or DB_ARCHIVE.stat().st_size == 0:
        run(["wget", "--continue", "--tries=3", "--timeout=30", "-O", DB_ARCHIVE, DB_URL])
    
    # Validasi integritas SHA256
    with open(DB_ARCHIVE, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    if file_hash == EXPECTED_SHA256:
        print(f"🔒 Verifikasi SHA256 valid: {file_hash[:16]}...")
    else:
        print(f"⚠️ Peringatan: Hash SHA256 berbeda ({file_hash[:16]}... vs {EXPECTED_SHA256[:16]}...)")
        
    run(["tar", "-xzf", DB_ARCHIVE, "-C", DB_DIR])

missing = [str(path) for path in required_db_files if not path.exists()]
assert not missing, f"Database tidak lengkap: {missing}"

bracken_models = sorted(path.name for path in DB_DIR.glob("database*mers.kmer_distrib"))
print("✅ Database Kraken2 NCBI Targeted Loci valid:", DB_DIR)
print("Model panjang read Bracken yang tersedia:", ", ".join(bracken_models) if bracken_models else "None")

---
## 2. Data publik, metadata, dan desain kelompok

BioProject **PRJNA637202** berisi 96 sampel feses bayi berusia 3 bulan dari studi PreventADALL. ENA mencatat data sebagai `AMPLICON`, `PCR`, `SINGLE`, platform `OXFORD_NANOPORE`, dan instrumen `MinION`.

File suplementer `MetadataPublish.xlsx` menghubungkan barcode ONT dengan mode persalinan: `0 = vaginal` dan `1 = sectio/sesar`. Barcode `BCxx` cocok dengan alias ENA `MergedSeqxx`, yang kemudian dipetakan ke accession SRR.

Untuk menjaga durasi workshop, kita memakai **subset seimbang 6 vaginal + 6 sesar**. Pemilihan hanya berdasarkan kecukupan jumlah read mentah, bukan berdasarkan komposisi mikrobioma. Ini adalah desain didaktik dan tidak dimaksudkan mereproduksi seluruh analisis publikasi.

> Mode persalinan adalah variabel kelompok, bukan diagnosis. Tanpa kovariat lengkap (misalnya antibiotik, pola menyusui, lokasi, dan faktor maternal), perbedaan yang ditemukan tetap berupa asosiasi eksploratif.


In [ ]:
# @title 2.1 Manifest subset seimbang dan unduh FASTQ (GitHub Release → Fallback Paralel ENA)
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from pathlib import Path

# Tautan arsip 12 FASTQ di GitHub Releases (jauh lebih cepat ~10 dtk di Colab jika sudah diunggah & repo Public)
RELEASE_ARCHIVE_URL = (
    "https://github.com/OmicsLite/Workshop-Long-Reads-Metagenomics/"
    "releases/download/v1.0-workshop-data/prjna637202_12samples_raw.tar.gz"
)

manifest_file = Path("sample_manifest.tsv")
if not manifest_file.exists():
    manifest_file = Path("hands-on/sample_manifest.tsv")

if manifest_file.exists():
    samples = pd.read_csv(manifest_file, sep="\t")
else:
    samples = pd.DataFrame([
        {"sample": "V01", "group": "Vaginal", "birth_code": 0, "barcode": "BC63", "alias": "MergedSeq63", "run": "SRR11923954", "read_count_ena": 45278, "bytes": 47935767, "subdir": "054"},
        {"sample": "V02", "group": "Vaginal", "birth_code": 0, "barcode": "BC73", "alias": "MergedSeq73", "run": "SRR11923943", "read_count_ena": 44293, "bytes": 47062268, "subdir": "043"},
        {"sample": "V03", "group": "Vaginal", "birth_code": 0, "barcode": "BC83", "alias": "MergedSeq83", "run": "SRR11923932", "read_count_ena": 43150, "bytes": 46042841, "subdir": "032"},
        {"sample": "V04", "group": "Vaginal", "birth_code": 0, "barcode": "BC76", "alias": "MergedSeq76", "run": "SRR11923940", "read_count_ena": 42560, "bytes": 45295782, "subdir": "040"},
        {"sample": "V05", "group": "Vaginal", "birth_code": 0, "barcode": "BC46", "alias": "MergedSeq46", "run": "SRR11923973", "read_count_ena": 42535, "bytes": 44492008, "subdir": "073"},
        {"sample": "V06", "group": "Vaginal", "birth_code": 0, "barcode": "BC77", "alias": "MergedSeq77", "run": "SRR11923939", "read_count_ena": 42091, "bytes": 43781623, "subdir": "039"},
        {"sample": "C01", "group": "Sesar", "birth_code": 1, "barcode": "BC49", "alias": "MergedSeq49", "run": "SRR11923970", "read_count_ena": 43527, "bytes": 45463513, "subdir": "070"},
        {"sample": "C02", "group": "Sesar", "birth_code": 1, "barcode": "BC89", "alias": "MergedSeq89", "run": "SRR11923926", "read_count_ena": 39632, "bytes": 40936549, "subdir": "026"},
        {"sample": "C03", "group": "Sesar", "birth_code": 1, "barcode": "BC19", "alias": "MergedSeq19", "run": "SRR11924003", "read_count_ena": 37036, "bytes": 38865995, "subdir": "003"},
        {"sample": "C04", "group": "Sesar", "birth_code": 1, "barcode": "BC84", "alias": "MergedSeq84", "run": "SRR11923931", "read_count_ena": 36843, "bytes": 38789861, "subdir": "031"},
        {"sample": "C05", "group": "Sesar", "birth_code": 1, "barcode": "BC86", "alias": "MergedSeq86", "run": "SRR11923929", "read_count_ena": 35293, "bytes": 36288277, "subdir": "029"},
        {"sample": "C06", "group": "Sesar", "birth_code": 1, "barcode": "BC66", "alias": "MergedSeq66", "run": "SRR11923951", "read_count_ena": 34498, "bytes": 36412478, "subdir": "051"},
    ])

samples["url"] = samples.apply(
    lambda row: f"https://ftp.sra.ebi.ac.uk/vol1/fastq/{row.run[:6]}/{str(row['subdir']).zfill(3)}/{row.run}/{row.run}_1.fastq.gz",
    axis=1,
)
samples["raw_fastq"] = samples["run"].map(lambda run_id: str(RAW_DIR / f"{run_id}.fastq.gz"))

assert samples.groupby("group").size().nunique() == 1, "Kelompok tidak seimbang"
assert samples["run"].is_unique and samples["barcode"].is_unique
display(samples[["sample", "group", "barcode", "alias", "run", "read_count_ena"]])

def all_fastq_valid():
    return all(
        Path(row.raw_fastq).exists() and Path(row.raw_fastq).stat().st_size > 10_000_000
        for row in samples.itertuples(index=False)
    )

# Lapis 1: Coba unduh 1 arsip cepat dari GitHub Releases
if not all_fastq_valid() and RELEASE_ARCHIVE_URL:
    release_tar = WORKDIR / "prjna637202_12samples_raw.tar.gz"
    print("🚀 Mencoba unduh cepat dari GitHub Releases...")
    res = subprocess.run(
        ["wget", "--continue", "--tries=2", "--timeout=30", "-q", "-O", str(release_tar), RELEASE_ARCHIVE_URL]
    )
    if res.returncode == 0 and release_tar.exists() and release_tar.stat().st_size > 1_000_000:
        print("📦 Mengekstrak arsip GitHub Release ke folder raw/...")
        run(["tar", "-xzf", release_tar, "-C", RAW_DIR])
    else:
        if release_tar.exists():
            release_tar.unlink()
        print("ℹ️ Arsip GitHub Release belum tersedia; otomatis beralih ke unduhan paralel dari ENA...")

# Lapis 2 (Fallback): Unduh paralel dari ENA + verifikasi ukuran byte eksak
def download_one(row):
    target = Path(row.raw_fastq)
    expected = int(row.bytes)
    if target.exists() and target.stat().st_size > 10_000_000:
        return row.sample, target.stat().st_size, "ready"
    subprocess.run(
        ["wget", "--continue", "--tries=5", "--timeout=45", "-q", "-O", str(target), row.url],
        check=True,
    )
    actual = target.stat().st_size
    assert actual == expected, f"Ukuran {row.sample} tidak cocok ({actual} vs {expected} bytes); jalankan ulang sel ini."
    return row.sample, actual, "downloaded"

if not all_fastq_valid():
    print("⬇️ Mengunduh FASTQ yang belum lengkap secara paralel (4 koneksi bersamaan)...")
done_count = 0
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(download_one, row) for row in samples.itertuples(index=False)]
    for fut in as_completed(futures):
        sample_id, size_b, status = fut.result()
        done_count += 1
        print(f"  [{done_count:02d}/{len(samples)}] ✅ {sample_id} ({size_b / 1e6:.1f} MB) — {status}")

print(f"\n✅ {len(samples)} FASTQ terunduh utuh & terverifikasi; {samples.groupby('group').size().to_dict()}")


In [ ]:
# @title 2.2 Validasi integritas FASTQ dan lihat contoh satu read
stats_raw_path = RES_QC_DIR / "seqkit_stats_raw.tsv"
with stats_raw_path.open("w") as handle:
    run(["seqkit", "stats", "-T", *samples["raw_fastq"]], stdout=handle)

stats_raw = pd.read_csv(stats_raw_path, sep="\t")
display(stats_raw)

# FASTQ terdiri dari 4 baris per read: header, sekuens, separator '+', dan kualitas.
import gzip
with gzip.open(samples.loc[0, "raw_fastq"], "rt") as handle:
    header, sequence, separator, quality = [handle.readline().strip() for _ in range(4)]
print("Header       :", header)
print("Panjang read :", len(sequence), "bp")
print("Sekuens awal :", sequence[:80] + "...")
print("Kualitas awal:", quality[:80] + "...")
assert separator == "+" and len(sequence) == len(quality)
print("\n✅ SeqKit dapat membaca semua file; struktur FASTQ valid.")

### 📖 Memahami Output di Atas: Anatomi FASTQ & Trik Phred Q-Score

Setiap 1 *read* di file FASTQ selalu terdiri dari **4 baris**: `@Header` (ID read), **Sekuens DNA** (`A, C, G, T`), pemisah `+`, dan **Skor Kualitas** dalam karakter ASCII (*Phred+33*).

* **Apa itu Phred Q-Score?** Ukuran peluang salah baca per basa: $Q = -10 \cdot \log_{10}(P_{\text{error}})$.
* **Trik cepat menghafalnya (tanpa logaritma!):** setiap nilai $Q$ naik **+10**, peluang error turun **10× lipat**:
  * **Q10** (ASCII `+`) = 1 salah dari 10 (**akurasi 90%**)
  * **Q20** (ASCII `5`) = 1 salah dari 100 (**akurasi 99%**)
  * **Q30** (ASCII `?`) = 1 salah dari 1.000 (**akurasi 99,9%**)

---
## 3. Quality control dan filtering

Skor Phred menghubungkan kualitas dengan probabilitas kesalahan: $Q=-10\log_{10}(P_{error})$. Q10 berarti estimasi error 10% per basa, bukan jaminan bahwa seluruh read 90% identik terhadap referensi.

Berdasarkan distribusi dataset ini (median sekitar 1,14 kb), filter workshop mempertahankan read 1.000–1.700 bp dengan mean Q≥10. Memaksakan cutoff 1.300 bp akan membuang hampir seluruh data. Rentang ini tetap **bukan detektor chimera**. Chimera PCR dapat memiliki panjang normal; analisis riset perlu langkah deteksi chimera/denoising yang tervalidasi untuk data ONT.

In [ ]:
# @title 3.1 Buat dan tampilkan laporan NanoPlot untuk sampel contoh (V01)
from IPython.display import HTML, display

nanoplot_dir = QC_DIR / "V01_raw"
report = nanoplot_dir / "NanoPlot-report.html"
if not report.exists():
    run([
        "NanoPlot", "--fastq", samples.loc[0, "raw_fastq"],
        "--outdir", nanoplot_dir, "--plots", "dot", "--threads", str(THREADS),
    ])

display(HTML(filename=str(report)))
shutil.copy2(report, RES_QC_DIR / "V01_NanoPlot-report.html")
print("Jika iframe tidak tampil otomatis di browser, buka dari panel Files:", report)

### 📖 Cara Membaca Laporan NanoPlot di Atas

Perhatikan **2 hal utama** pada grafik distribusi panjang dan kualitas di atas:
1. **Puncak panjang read di ~1.140 bp (1,14 kb)**: Ini **bukan** DNA terdegradasi! Primer PCR pada studi PRJNA637202 memang menargetkan wilayah **V3–V9** (~1,14 kb), bukan full-length V1–V9 (~1,5 kb).
2. **Rata-rata kualitas di kisaran Q10–Q14**: Wajar untuk kimia Nanopore **R9.4.1** (tahun 2020). Kimia modern **R10.4.1** saat ini sudah mencapai >Q20 (>99%). Karena itu, batas filter kita nanti diatur realistis di **Q ≥ 10** dan **1.000–1.700 bp**.

In [ ]:
# @title 3.2 Filter semua sampel dengan Chopper
filtered_paths = []
for row in samples.itertuples(index=False):
    output = FILTERED_DIR / f"{row.run}.filtered_L{MIN_LENGTH}-{MAX_LENGTH}_Q{MIN_MEAN_Q}.fastq.gz"
    filtered_paths.append(str(output))
    if not output.exists() or output.stat().st_size == 0:
        command = (
            f"gunzip -c {row.raw_fastq} | "
            f"chopper -q {MIN_MEAN_Q} -l {MIN_LENGTH} --maxlength {MAX_LENGTH} | "
            f"gzip -c > {output}"
        )
        run(["bash", "-o", "pipefail", "-c", command])
    assert output.stat().st_size > 0, f"Tidak ada read lolos untuk {row.run}"

samples["filtered_fastq"] = filtered_paths

stats_filtered_path = RES_QC_DIR / "seqkit_stats_filtered.tsv"
with stats_filtered_path.open("w") as handle:
    run(["seqkit", "stats", "-T", *samples["filtered_fastq"]], stdout=handle)
stats_filtered = pd.read_csv(stats_filtered_path, sep="\t")

qc_summary = samples[["sample", "group", "run"]].copy()
qc_summary["raw_reads"] = stats_raw["num_seqs"].to_numpy()
qc_summary["filtered_reads"] = stats_filtered["num_seqs"].to_numpy()
qc_summary["retained_pct"] = 100 * qc_summary["filtered_reads"] / qc_summary["raw_reads"]
qc_summary.to_csv(RES_QC_DIR / "qc_summary.tsv", sep="\t", index=False)
display(qc_summary.style.format({"retained_pct": "{:.1f}%"}))
print("✅ Filtering selesai. Retensi sangat rendah adalah sinyal untuk memeriksa plot panjang/kualitas, bukan sekadar melanjutkan.")

### 📖 Memahami Hasil Filtering (`Chopper`) & Mitos Chimera

Bayangkan `Chopper` seperti **ayakan saringan biji kopi**: hanya meloloskan read dengan ukuran pas (`1.000–1.700 bp`) dan mutu memadai (`Q ≥ 10`).
* **Retensi ~70–85%** adalah angka yang sehat pada data R9.4.1. Jika batas bawah dipaksa ke `1.300 bp`, >90% data valid akan terbuang!
* **Waspadai Mitos**: Filter panjang membuang fragmen pendek (*primer-dimer*) dan terlalu panjang (*concatemer*), tetapi **tidak otomatis membuang molekul PCR chimera** karena banyak chimera berukuran persis normal (~1,1–1,5 kb).

### Checkpoint diskusi

- Apakah puncak panjang read berada dekat 1.500 bp?
- Berapa persen read yang hilang setelah filter, dan parameter mana yang mungkin paling berpengaruh?
- Apa trade-off jika `MIN_MEAN_Q` dinaikkan?

> Filtering dapat mengubah komposisi bila kualitas berbeda antar takson. Karena itu, simpan tabel retensi dan laporkan parameter yang dipakai.

---
## 4. Klasifikasi taksonomi dengan Kraken2

Kraken2 memetakan *minimizer* dari k-mer read ke takson dan menggunakan lowest common ancestor (LCA) ketika bukti tidak unik. Parameter `--confidence 0.01` mensyaratkan sedikitnya 1% k-mer informatif mendukung takson/keturunannya; ini secara efektif mengurangi panggilan acak (*spurious assignments*).

Database **NCBI Targeted Loci 16S/18S** dari ONT EPI2ME Labs dibangun dari strain tipe (*type strain*) resmi NCBI, memungkinkan identifikasi taksonomi hingga tingkat **Spesies** (`rank = S`) di samping tingkat **Genus** (`rank = G`).

> **Catatan Ilmiah Penting:** Kehadiran label taksonomi tingkat spesies pada laporan Kraken2 **bukan jaminan mutlak 100% kepastian spesies**. Beberapa spesies berkerabat dekat (misal dalam genus *Bacillus* atau *Escherichia/Shigella*) memiliki sekuens 16S yang nyaris identik. Oleh karena itu, kita menganalisis data pada **tingkat Spesies** sekaligus menyediakan rekapitulasi pada **tingkat Genus** sebagai kontrol metodologis.

In [ ]:
# @title 4.1 Jalankan Kraken2 dan simpan output per-read + report
kraken_outputs, kraken_reports = [], []

for row in samples.itertuples(index=False):
    output = RES_CLASS_PER_READ_DIR / f"{row.run}.kraken.tsv"
    report = RES_CLASS_REPORTS_DIR / f"{row.run}.k2report"
    kraken_outputs.append(str(output))
    kraken_reports.append(str(report))
    run([
        "kraken2", "--db", DB_DIR, "--threads", str(THREADS),
        "--confidence", str(KRAKEN_CONFIDENCE),
        "--output", output, "--report", report, row.filtered_fastq,
    ])
    assert report.exists() and report.stat().st_size > 0, f"Report gagal: {report}"

samples["kraken_output"] = kraken_outputs
samples["kraken_report"] = kraken_reports
print("✅ Klasifikasi selesai.")

In [ ]:
# @title 4.2 Ringkas classified vs unclassified
def read_kraken_report(path):
    columns = ["percent", "clade_reads", "direct_reads", "rank", "taxid", "name"]
    df = pd.read_csv(path, sep="\t", names=columns)
    df["name"] = df["name"].str.strip()
    return df

classification_rows = []
for row in samples.itertuples(index=False):
    report = read_kraken_report(row.kraken_report)
    unclassified = int(report.loc[report["rank"] == "U", "clade_reads"].sum())
    classified = int(report.loc[report["rank"] == "R", "clade_reads"].sum())
    total = classified + unclassified
    classification_rows.append({
        "sample": row.sample, "group": row.group, "run": row.run, "classified": classified,
        "unclassified": unclassified, "classified_pct": 100 * classified / total,
    })

classification_summary = pd.DataFrame(classification_rows)
classification_summary.to_csv(RES_CLASS_DIR / "classification_summary.tsv", sep="\t", index=False)
display(classification_summary.style.format({"classified_pct": "{:.1f}%"}))

### 📖 Memahami Hasil Kraken2: Cara Kerja *Minimizer k-mer* & LCA

`Kraken2` memotong tiap read menjadi potongan pendek (*k-mer*, $k=35$) lalu mencocokkannya ke tabel hash database resmi **ONT EPI2ME NCBI Targeted Loci** (~8,6 MB).
* **Analogi Detektif Kamus yang Jujur (LCA):** Jika petunjuk k-mer unik untuk satu spesies $\rightarrow$ ditempatkan di **Spesies (`S`)**. Jika cocok dengan beberapa spesies berkerabat dekat $\rightarrow$ Kraken2 jujur naik ke leluhur bersama terdekat (**Genus `G` / Famili `F`**) daripada asal menebak.
* Flag `--confidence 0.01` mensyaratkan minimal 1% k-mer informatif mendukung cabang tersebut guna menekan penugasan acak (*false positive*).

---
## 5. Tabel kelimpahan taksonomi (Spesies & Genus)

Untuk setiap takson (baik Spesies `rank = S` maupun Genus `rank = G`), nilai `clade_reads` mencakup seluruh read yang ditempatkan pada takson tersebut maupun varian turunannya. 

Kelimpahan relatif dinormalisasi terhadap total hitungan takson yang terdeteksi di sampel. Ini adalah **profil komposisional**: nilai persentase mencerminkan proporsi sekuens di dalam pustaka, bukan konsentrasi sel absolut di usus.

Untuk menyingkirkan panggilan tingkat rendah (*background noise*), kita menerapkan ambang $\ge 10$ reads per sampel untuk matriks utama **Genus**, serta $\ge 1$ read untuk eksplorasi **Spesies terarah**.

In [ ]:
# @title 5.1 Bentuk matriks hitungan taksonomi (Genus & Spesies)
import numpy as np
MIN_GENUS_READS = 10
MIN_SPECIES_READS = 1

def build_abundance_matrix(samples_df, target_rank="G", min_reads=10):
    series_dict = {}
    for row in samples_df.itertuples(index=False):
        report = read_kraken_report(row.kraken_report)
        subset = report.loc[report["rank"] == target_rank, ["name", "clade_reads"]].copy()
        subset = subset[subset["clade_reads"] >= min_reads]
        series_dict[row.sample] = subset.groupby("name")["clade_reads"].sum()
    counts = pd.DataFrame(series_dict).reindex(columns=samples_df["sample"]).fillna(0).astype(int).sort_index()
    return counts

# Matriks Utama (Kedalaman Tinggi): Tingkat Genus (rank = G)
genus_counts = build_abundance_matrix(samples, target_rank="G", min_reads=MIN_GENUS_READS)
genus_counts.to_csv(RES_ABUND_TABLES_DIR / "genus_counts.tsv", sep="\t")

genus_relative = genus_counts.div(genus_counts.sum(axis=0), axis=1).fillna(0)
genus_relative.to_csv(RES_ABUND_TABLES_DIR / "genus_relative_abundance.tsv", sep="\t")

# Matriks Eksploratif: Tingkat Spesies (rank = S)
species_counts = build_abundance_matrix(samples, target_rank="S", min_reads=MIN_SPECIES_READS)
species_counts.to_csv(RES_ABUND_TABLES_DIR / "species_counts.tsv", sep="\t")

species_relative = species_counts.div(species_counts.sum(axis=0), axis=1).fillna(0)
species_relative.to_csv(RES_ABUND_TABLES_DIR / "species_relative_abundance.tsv", sep="\t")

# Diagnostik kedalaman Genus vs Spesies (menunjukkan perilaku LCA Kraken2 pada R9.4.1)
depth_summary = pd.DataFrame({
    "group": samples.set_index("sample")["group"],
    "genus_reads": genus_counts.sum(axis=0),
    "species_reads": species_counts.sum(axis=0),
})
depth_summary["species_to_genus_pct"] = (100 * depth_summary["species_reads"] / depth_summary["genus_reads"].replace(0, np.nan)).round(1)
print(f"📊 Takson terdeteksi:")
print(f"   • Genus (rank G, ≥{MIN_GENUS_READS} read): {genus_counts.shape[0]} genus unik")
print(f"   • Spesies terarah (rank S, ≥{MIN_SPECIES_READS} read): {species_counts.shape[0]} spesies unik\n")
depth_summary.to_csv(RES_ABUND_TABLES_DIR / "lca_depth_summary.tsv", sep="\t")
display(depth_summary)


### 📖 Momen "Aha!": Kenapa `species_reads` Jauh Lebih Sedikit dari `genus_reads`?

Perhatikan tabel diagnostik di atas! Pada sebagian sampel (misalnya kelompok Sesar), ribuan read berhenti di tingkat **Genus**, sementara hanya sebagian kecil yang lolos hingga **Spesies**.
* **Ini BUKAN error**, melainkan bukti transparansi algoritma **LCA Kraken2** pada data R9.4.1: error *indel* homopolimer dan tingginya kemiripan 16S antarspesies dekat (seperti *E. coli* vs *Shigella*) membuat banyak read ditahan secara aman di tingkat Genus.
* Untuk penelitian lanjutan yang menargetkan akurasi spesies maksimal pada data 16S Nanopore, gunakan perangkat lunak berbasis *Expectation-Maximization* (**`Emu`**, Curry et al., 2022).

In [ ]:
# @title 5.2 Visualisasi komposisi taksonomi dominan (Genus & Spesies)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
GROUP_COLORS = {"Vaginal": "#4C78A8", "Sesar": "#E45756"}
TOP_N = 12

def plot_stacked_composition(counts_df, rel_df, rank_label, out_filename, top_n=12):
    if counts_df.empty:
        print(f"⚠️ Tidak ada data untuk {rank_label}.")
        return
    top_taxa = counts_df.sum(axis=1).nlargest(top_n).index
    plot_table = rel_df.loc[top_taxa].T.copy()
    plot_table[f"{rank_label} lain"] = (1 - plot_table.sum(axis=1)).clip(lower=0)

    ax = (100 * plot_table).plot(kind="bar", stacked=True, figsize=(14, 5.5), colormap="tab20")
    ax.set_ylabel("Kelimpahan relatif (%)")
    ax.set_xlabel("Sampel (V = Vaginal; C = Sesar)")
    ax.set_title(f"Komposisi {min(top_n, len(top_taxa))} {rank_label} Dominan (NCBI Targeted Loci)")
    ax.legend(title=rank_label, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.savefig(RES_ABUND_FIGS_DIR / out_filename, dpi=300)
    plt.show()

# 1. Grafik Utama: Tingkat Genus (kedalaman tinggi & stabil lintas semua sampel)
plot_stacked_composition(genus_counts, genus_relative, "Genus", "genus_composition_barplot.png", top_n=TOP_N)

# 2. Grafik Eksploratif: Tingkat Spesies Terarah (putative species)
plot_stacked_composition(species_counts, species_relative, "Spesies", "species_composition_barplot.png", top_n=TOP_N)


### 📖 Cara Membaca Grafik Komposisi & Jebakan "Kue Pizza" (Data Komposisional)

Grafik batang bertumpuk di atas menampilkan **kelimpahan relatif (total selalu 100%)**, bukan jumlah sel bakteri absolut!
* **Analogi Potongan Kue Pizza**: Jika satu bakteri (mis. *Bacteroides*) turun drastis, persentase bakteri lain (mis. *Bifidobacterium*) akan tampak melonjak besar di grafik—**meskipun jumlah sel fisiknya tetap atau bahkan ikut menurun!**
* Untuk mengetahui jumlah sel absolut di usus, diperlukan pengukuran tambahan di lab basah seperti *qPCR* 16S atau *flow cytometry*.

---
## 6. Krona interaktif

Utilitas **Krona Plot** merender hierarki taksonomi dalam diagram lingkaran konsentris bertingkat (*multi-layered pie chart*). Kita mengekstrak jalur taksonomi dari laporan Kraken2 NCBI Targeted Loci menjadi format input `ktImportText`. Diagram ini memungkinkan eksplorasi intuitif dari tingkat Domain $\rightarrow$ Phylum $\rightarrow$ Class $\rightarrow$ Order $\rightarrow$ Family $\rightarrow$ Genus $\rightarrow$ Species secara interaktif di browser.

In [ ]:
# @title 6.1 Buat Krona dari hierarki taksonomi NCBI
def kraken_report_to_krona_text(report_path, output_path):
    columns = ["percent", "clade_reads", "direct_reads", "rank", "taxid", "name"]
    report = pd.read_csv(report_path, sep="\t", names=columns, keep_default_na=False)
    lineage = {}
    with Path(output_path).open("w") as handle:
        for item in report.itertuples(index=False):
            raw_name = str(item.name)
            depth = (len(raw_name) - len(raw_name.lstrip())) // 2
            lineage[depth] = raw_name.strip()
            lineage = {level: name for level, name in lineage.items() if level <= depth}
            if item.rank not in ("U", "R") and int(item.direct_reads) > 0:
                path = [lineage[level] for level in sorted(lineage) if lineage[level] != "root"]
                handle.write(str(int(item.direct_reads)) + "\t" + "\t".join(path) + "\n")

krona_files = []
for row in samples.itertuples(index=False):
    krona_input = RES_KRONA_TSV_DIR / f"{row.run}.krona.tsv"
    output_html = RES_KRONA_HTML_DIR / f"{row.run}.krona.html"
    kraken_report_to_krona_text(row.kraken_report, krona_input)
    assert krona_input.stat().st_size > 0, f"Input Krona kosong: {krona_input}"
    run(["ktImportText", krona_input, "-o", output_html])
    krona_files.append(str(output_html))

samples["krona_html"] = krona_files
print("✅ Krona selesai diproduksi untuk seluruh sampel.")

In [ ]:
# @title 6.2 Tampilkan Krona untuk satu sampel contoh (V01)
import base64

v01_krona = Path(samples.loc[samples["sample"] == "V01", "krona_html"].iloc[0])
encoded = base64.b64encode(v01_krona.read_bytes()).decode("ascii")
display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="650" style="border:1px solid #ddd; border-radius:6px;"></iframe>'))
print("💡 Tips: Jika diagram tidak tampil interaktif di browser Anda, unduh file ini dari panel Files kiri lalu buka di Chrome/Safari:", v01_krona)


### 📖 Cara Menjelajahi Krona Plot Interaktif

* **Lingkaran terdalam** menunjukkan tingkat **Domain & Filum** (*Actinomycetota* / dahulu *Actinobacteria*, *Bacteroidota* / *Bacteroidetes*, *Bacillota* / *Firmicutes* — sesuai tata nama resmi **ICNP 2021**).
* **Semakin ke luar**, cabang makin rinci ke **Famili $\rightarrow$ Genus $\rightarrow$ Spesies**. Klik sektor mana saja untuk memperbesar (*zoom-in*) dan melihat proporsinya.

---
## 7. Ekologi Komunitas (Diversity Analysis)

Standardisasi kedalaman sekuensing melalui **rarefaction** penting agar perbandingan keanekaragaman tidak bias oleh kedalaman sekuensing yang berbeda.

> **Mengapa rarefaction & uji diversity dijalankan pada matriks Genus?**  
> Seperti terlihat pada tabel diagnostik Sel 5.1, algoritma LCA Kraken2 pada data R9.4.1 menempatkan **ribuan read per sampel** pada tingkat **Genus**, sedangkan hanya sebagian kecil (kadang <10 read pada sampel tertentu) yang lolos hingga tingkat Spesies. Menjalankan rarefaction pada matriks **Genus** menjaga kedalaman sampling tetap tinggi dan statistik tetap kokoh, sementara temuan tingkat **Spesies** tetap kita rangkum secara paralel di Sel 7.3!

- **Observed genera:** Jumlah genus unik yang terdeteksi setelah rarefaction.
- **Shannon index ($H'$):** $H'=-\sum p_i\ln(p_i)$; mencerminkan kekayaan (*richness*) sekaligus kemerataan (*evenness*).
- **Simpson index ($1-D$):** Probabilitas dua read acak berasal dari genus berbeda.
- **Bray–Curtis dissimilarity:** Jarak ketidaksamaan komposisi antar-sampel (0 = identik, 1 = tidak berbagi takson sama sekali).


In [ ]:
# @title 7.1 Rarefaction, alpha diversity (Genus), dan perbandingan kelompok
import numpy as np
from scipy.stats import mannwhitneyu

depths = genus_counts.sum(axis=0)
RAREFACTION_DEPTH = int(depths.min())
assert RAREFACTION_DEPTH > 0, "Kedalaman genus nol."
print(f"Kedalaman rarefaction genus: {RAREFACTION_DEPTH:,} reads/sampel")

rng = np.random.default_rng(SEED)
rarefied = pd.DataFrame(index=genus_counts.index)
for sample in genus_counts.columns:
    counts = genus_counts[sample].to_numpy(dtype=np.int64)
    rarefied[sample] = rng.multivariate_hypergeometric(counts, RAREFACTION_DEPTH)
assert (rarefied.sum(axis=0) == RAREFACTION_DEPTH).all()
rarefied.to_csv(RES_ALPHA_DIR / "genus_counts_rarefied.tsv", sep="\t")

def alpha_metrics(counts):
    counts = np.asarray(counts)
    counts = counts[counts > 0]
    proportions = counts / counts.sum()
    return {
        "Observed genera": len(counts),
        "Shannon": -np.sum(proportions * np.log(proportions)),
        "Simpson (1-D)": 1 - np.sum(proportions ** 2),
    }

alpha = pd.DataFrame({sample: alpha_metrics(rarefied[sample]) for sample in rarefied}).T
alpha.index.name = "sample"
alpha = alpha.reset_index().merge(samples[["sample", "group", "run"]], on="sample")
alpha.to_csv(RES_ALPHA_DIR / "alpha_diversity_genus.tsv", sep="\t", index=False)
display(alpha.round(3))

test_rows = []
metrics = ["Observed genera", "Shannon", "Simpson (1-D)"]
for metric in metrics:
    vaginal = alpha.loc[alpha.group == "Vaginal", metric]
    sesar = alpha.loc[alpha.group == "Sesar", metric]
    statistic, p_value = mannwhitneyu(vaginal, sesar, alternative="two-sided")
    test_rows.append({
        "metric": metric,
        "median_vaginal": vaginal.median(),
        "median_sesar": sesar.median(),
        "U": statistic,
        "p_value": p_value,
    })

alpha_tests = pd.DataFrame(test_rows)
alpha_tests.to_csv(RES_ALPHA_DIR / "alpha_group_tests.tsv", sep="\t", index=False)
display(alpha_tests.round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, metrics):
    sns.boxplot(data=alpha, x="group", y=metric, hue="group", palette=GROUP_COLORS, legend=False, width=0.5, ax=ax)
    sns.stripplot(data=alpha, x="group", y=metric, color="#222222", size=6, ax=ax)
    p_val = alpha_tests.loc[alpha_tests.metric == metric, "p_value"].iloc[0]
    ax.set_title(f"{metric}\nMann–Whitney p = {p_val:.4f}")
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(RES_ALPHA_DIR / "alpha_diversity_boxplot.png", dpi=300)
plt.show()


### 📖 Panduan Praktis Membaca Rarefaction & Indeks Alpha Diversity

* **Rarefaction ("Menjaring Ikan di Dua Kolam")**: Menyamakan jumlah "tarikan jaring" (`SEED = 42`) agar sampel yang punya lebih banyak read tidak otomatis terlihat lebih kaya jenis.

#### 📏 *Rule of Thumb* Membaca Setiap Indeks Alpha Diversity

| Indeks | Rentang Umum | Jika Nilainya **TINGGI** artinya... | Jika Nilainya **RENDAH** artinya... | Patokan Ringan (*Rule of Thumb*) |
|---|---|---|---|---|
| **Observed Genera** (*Richness*) | `1 – ratusan` | **Banyak jenis** genus bakteri berbeda hadir di sampel. | Hanya **segelintir jenis** genus yang terdeteksi. | Bayi 3 bulan wajar di kisaran **15–40 genus** (dewasa: 80–150+). |
| **Shannon Index ($H'$)** | `0 – ~4+` | Jenis banyak **SEKALIGUS** porsinya **merata** (tidak ada monopoli). | Jenis sedikit **ATAU** 1–2 bakteri mendominasi hampir seluruh sampel. | `< 1,0` = sangat didominasi 1–2 takson; `1,0–2,0` = moderat (khas bayi); `> 2,5` = sangat beragam & merata. |
| **Simpson ($1 - D$)** | `0 – 1` | Mendekati **1** (mis. `0,7–0,9`): beragam & merata; peluang 2 read acak berbeda genus sangat besar. | Mendekati **0** (mis. `< 0,4`): 1 genus **memonopoli** komunitas. | Lebih sensitif terhadap bakteri paling dominan dibanding takson langka. |

> **🍎 Analogi Dua Keranjang Buah:** Keranjang A dan B sama-sama punya **5 jenis buah** (*Observed* sama = 5). Namun Keranjang A isinya 96 apel + 1 jeruk + 1 mangga + 1 pisang + 1 anggur, sedangkan Keranjang B isinya 20 tiap buah. Keranjang B jauh lebih merata (*Evenness* tinggi) $\rightarrow$ **Shannon & Simpson ($1-D$) Keranjang B jauh lebih tinggi!**

#### 🔄 Apakah Ada Indeks yang Saling Bertolak Belakang?
1. **Hati-hati `Simpson D` vs `Simpson (1 - D)` di jurnal!**
   * **`Simpson D` (Dominansi)** berlawanan arah: makin besar angkanya = makin *monopoli* (makin **tidak** beragam).
   * Di notebook ini kita sengaja memakai **`Simpson (1 - D)`** sehingga **ketiga indeks kita SEARAH**: **makin tinggi angka = makin beragam**.
2. **Observed tinggi tapi Shannon/Simpson rendah — kok bisa?**
   * Bisa! Artinya sampel punya banyak genus (mis. 35 genus), **tetapi 1 genus memonopoli 90% sampel** sementara 34 genus lain hanya muncul sangat sedikit sebagai "figuran".
3. **⚠️ Mitos "Diversity Tinggi Selalu Lebih Sehat"**:
   * **Tidak selalu!** Pada **bayi ASI usia 3 bulan**, Alpha Diversity yang rendah–moderat justru sering **sehat secara fisiologis** karena usus bayi idealnya didominasi *Bifidobacterium* pencerna ASI (HMO). Contoh lain: mikrobioma vagina sehat justru ber-diversity rendah karena didominasi *Lactobacillus*. Selalu baca angka bersama **konteks usia & organ**!

In [ ]:
# @title 7.2 Bray–Curtis, PCoA, dan PERMANOVA (Tingkat Genus)
from scipy.spatial.distance import pdist, squareform

sample_by_genus = rarefied.T.loc[samples["sample"]]
distance = squareform(pdist(sample_by_genus, metric="braycurtis"))
beta = pd.DataFrame(distance, index=sample_by_genus.index, columns=sample_by_genus.index)
beta.to_csv(RES_BETA_DIR / "bray_curtis_genus.tsv", sep="\t")

# PCoA (classical multidimensional scaling)
n = len(beta)
J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (distance ** 2) @ J
eigenvalues, eigenvectors = np.linalg.eigh(B)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
positive = eigenvalues > 1e-12
coords = eigenvectors[:, positive] * np.sqrt(eigenvalues[positive])
explained = eigenvalues[positive] / eigenvalues[positive].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(beta, vmin=0, vmax=1, cmap="Reds", square=True, ax=axes[0])
axes[0].set_title("Bray–Curtis dissimilarity (Genus)")

group_lookup = samples.set_index("sample")["group"]
for group, color in GROUP_COLORS.items():
    mask = np.array([group_lookup[s] == group for s in beta.index])
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=80, label=group, color=color)
for label, x, y in zip(beta.index, coords[:, 0], coords[:, 1]):
    axes[1].annotate(label, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=9)
axes[1].set_xlabel(f"PCoA1 ({explained[0] * 100:.1f}%)")
axes[1].set_ylabel(f"PCoA2 ({explained[1] * 100:.1f}%)")
axes[1].set_title("PCoA Bray–Curtis (Tingkat Genus)")
axes[1].legend(title="Persalinan")
plt.tight_layout()
plt.savefig(RES_BETA_DIR / "beta_diversity_genus_pcoa.png", dpi=300)
plt.show()

def permanova_one_factor(distance_matrix, labels, permutations=999, seed=42):
    D = np.asarray(distance_matrix, dtype=float)
    labels = np.asarray(labels)
    n = len(labels)
    G = -0.5 * (np.eye(n) - np.ones((n, n))/n) @ (D ** 2) @ (np.eye(n) - np.ones((n, n))/n)
    groups = np.unique(labels)

    def pseudo_f(current_labels):
        X = np.column_stack([current_labels == g for g in groups]).astype(float)
        H = X @ np.linalg.pinv(X.T @ X) @ X.T
        ss_between = np.trace(H @ G)
        ss_total = np.trace(G)
        return (ss_between / (len(groups) - 1)) / ((ss_total - ss_between) / (n - len(groups)))

    observed = pseudo_f(labels)
    rng = np.random.default_rng(seed)
    permuted = np.array([pseudo_f(rng.permutation(labels)) for _ in range(permutations)])
    p_value = (1 + np.sum(permuted >= observed)) / (permutations + 1)
    return observed, p_value

labels = samples.set_index("sample").loc[beta.index, "group"].to_numpy()
permanova_f, permanova_p = permanova_one_factor(distance, labels, permutations=999, seed=SEED)
permanova_result = pd.DataFrame([{"pseudo_F": permanova_f, "p_value": permanova_p,
                                 "permutations": 999, "n_per_group": 6}])
permanova_result.to_csv(RES_BETA_DIR / "permanova_genus.tsv", sep="\t", index=False)
display(permanova_result.round(4))

### 📖 Panduan Praktis Membaca Beta Diversity (Bray–Curtis, PCoA & PERMANOVA)

Kalau **Alpha Diversity** melihat isi *di dalam satu sampel*, **Beta Diversity** membandingkan *seberapa beda isi antara dua sampel atau dua kelompok*.

#### 📏 *Rule of Thumb* Membaca Metrik Beta Diversity

| Metrik | Rentang Nilai | Jika Nilainya **TINGGI** artinya... | Jika Nilainya **RENDAH** artinya... | Cara Cepat Membacanya |
|---|---|---|---|---|
| **Bray–Curtis Dissimilarity** | `0 – 1` | Mendekati **`1`** (mis. `> 0,7`): komposisi dua sampel **SANGAT BERBEDA** (`1,0` = tak berbagi 1 genus pun). | Mendekati **`0`** (mis. `< 0,3`): komposisi dua sampel **SANGAT MIRIP** (`0,0` = identik persis). | **⚠️ Kebalikan dari "kemiripan"!** Makin tinggi angka = makin **BEDA**. |
| **PCoA (% di Sumbu X & Y)** | `0% – 100%` | Total PCoA1 + PCoA2 besar (mis. `> 50%`): peta 2D sudah cukup jujur mewakili jarak asli. | Persentase kecil (`< 20%`): data sangat kompleks; peta 2D hanya menangkap sebagian variasi. | Bayangkan **peta jarak antar-kota**: titik berdekatan = mirip, berjauhan = beda. |
| **PERMANOVA (`Pseudo-F` & `p`)** | `F > 0`, `p: 0–1` | `Pseudo-F` makin besar & **`p < 0,05`**: perbedaan antar-kelompok lebih besar daripada variasi acak di dalam kelompok. | `Pseudo-F` kecil (~1) & **`p ≥ 0,05`**: belum cukup bukti statistik kedua kelompok terpisah (bisa karena memang mirip atau $n$ kecil). | Selalu baca bersama plot PCoA karena `p < 0,05` bisa dipicu perbedaan pusat (*centroid*) maupun sebaran (*dispersi*). |

> **🔄 Hubungan Alpha vs Beta Diversity (Bisa Tidak Sejalan!):**
> Dua kelompok (mis. Vaginal vs Sesar) bisa saja punya **Alpha Diversity yang SAMA** (*sama-sama punya ~30 genus, p > 0,05*), **tetapi Beta Diversity-nya BERBEDA JAUH** — karena *jenis bakterinya berbeda total* (misalnya satu didominasi *Bacteroides*, satunya didominasi *Clostridium/Veillonella*)! Karena itu, Alpha dan Beta Diversity **wajib selalu dibaca berpasangan**.

### 7.3 Takson yang berbeda antar-kelompok (Uji Hipotesis Eksploratif)

Kita membandingkan kelimpahan relatif setiap **Genus** antara kelompok persalinan Vaginal vs Sesar menggunakan uji non-parametrik Mann–Whitney U, lalu mengoreksi pengujian hipotesis multipel dengan metode **Benjamini–Hochberg False Discovery Rate (FDR)**. Di bawahnya, kita juga menampilkan ringkasan eksploratif deteksi **Spesies terarah (*putative species*)** lintas kedua kelompok.


In [ ]:
# @title 7.3 Uji Genus (Mann–Whitney + BH-FDR) & Ringkasan Eksplorasi Spesies
from scipy.stats import mannwhitneyu

def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = np.minimum.accumulate((ranked * len(ranked) / np.arange(1, len(ranked)+1))[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted

relative_rarefied = rarefied.div(rarefied.sum(axis=0), axis=1)
v_samples = samples.loc[samples.group == "Vaginal", "sample"].tolist()
c_samples = samples.loc[samples.group == "Sesar", "sample"].tolist()

# 1. Uji Statistik Utama: Tingkat Genus
rows = []
for genus in relative_rarefied.index:
    v = relative_rarefied.loc[genus, v_samples]
    c = relative_rarefied.loc[genus, c_samples]
    statistic, p_value = mannwhitneyu(v, c, alternative="two-sided")
    rows.append({
        "genus": genus,
        "median_vaginal_pct": 100 * v.median(),
        "median_sesar_pct": 100 * c.median(),
        "prevalence_vaginal": (v > 0).mean(),
        "prevalence_sesar": (c > 0).mean(),
        "U": statistic,
        "p_value": p_value,
    })
genus_tests = pd.DataFrame(rows)
genus_tests["q_value_BH"] = benjamini_hochberg(genus_tests["p_value"])
genus_tests = genus_tests.sort_values(["q_value_BH", "p_value"])
genus_tests.to_csv(RES_DIFF_DIR / "genus_group_tests.tsv", sep="\t", index=False)
print("=== Uji Perbedaan Kelimpahan Tingkat Genus (Koreksi BH-FDR) ===")
display(genus_tests.head(15).round(4))
print("Jumlah genus dengan q < 0.05 (FDR):", int((genus_tests.q_value_BH < 0.05).sum()))

# 2. Ringkasan Eksploratif: Spesies Terarah (Putative Species)
if not species_counts.empty:
    sp_rows = []
    for sp in species_counts.index:
        v_cnt = species_counts.loc[sp, v_samples]
        c_cnt = species_counts.loc[sp, c_samples]
        sp_rows.append({
            "species": sp,
            "total_reads_vaginal": int(v_cnt.sum()),
            "total_reads_sesar": int(c_cnt.sum()),
            "prevalence_vaginal": (v_cnt > 0).mean(),
            "prevalence_sesar": (c_cnt > 0).mean(),
        })
    species_summary = pd.DataFrame(sp_rows).sort_values(
        by=["total_reads_vaginal", "total_reads_sesar"], ascending=False
    )
    species_summary.to_csv(RES_DIFF_DIR / "species_group_summary.tsv", sep="\t", index=False)
    print("\n=== Ringkasan Eksplorasi Spesies Terarah (Putative Species, NCBI Targeted Loci) ===")
    display(species_summary.head(15).round(3))


### 📖 Memahami `q_value_BH` (FDR) & Batas Interpretasi Spesies

* **Kenapa wajib melihat `q_value_BH`, bukan `p_value` mentah?** Saat kita menguji puluhan takson sekaligus, sebagian bisa tampak $p < 0,05$ murni karena kebetulan acak (*multiple testing problem*). Koreksi **Benjamini–Hochberg (FDR)** menyaring kebetulan tersebut menjadi `$q$-value` (bermakna jika $q < 0,05$).
* **Kehati-hatian Klinis**: Tabel spesies terarah di atas membantu eksplorasi biologis (mis. *Bacteroides fragilis* pada bayi lahir vaginal), namun **deteksi DNA 16S $\neq$ bakteri hidup $\neq$ infeksi klinis**, dan hasil subset 6+6 ini adalah **asosiasi eksploratif**, bukan diagnosis medis.

## 8. Interpretasi yang bertanggung jawab

Gunakan pola **observasi → angka → ketidakpastian → batas kesimpulan**.

Contoh: “Median Shannon berbeda antara subset vaginal dan sesar setelah rarefaction. Uji Mann–Whitney menghasilkan p = … . Karena hanya enam sampel per kelompok dan kovariat penting tidak dikontrol, hasil ini adalah asosiasi eksploratif, bukan efek kausal mode persalinan.”

Hindari kesimpulan berikut:

- “Sesar menyebabkan genus X hilang”—desain ini observasional.
- “p > 0,05 membuktikan kedua kelompok sama”—daya statistik subset kecil.
- “PERMANOVA signifikan berarti centroid pasti berbeda”—perbedaan dispersi juga dapat memengaruhi hasil.
- “Genus X adalah patogen”—16S tidak membuktikan strain, viabilitas, atau kausalitas.
- “Kelimpahan relatif sama dengan jumlah sel”—PCR, copy number 16S, ekstraksi, dan klasifikasi menimbulkan bias.

### Pertanyaan diskusi

1. Apakah retensi filtering sebanding antara kedua kelompok?
2. Apakah boxplot alpha diversity menunjukkan overlap besar?
3. Apakah sampel mengelompok pada PCoA, dan adakah outlier?
4. Apakah hasil genus tetap signifikan setelah koreksi BH, dan apakah pola spesies terarahnya sejalan?
5. Faktor perancu apa yang perlu dimasukkan dalam studi penuh?
6. Mengapa subset 6+6 tidak boleh dianggap mereproduksi publikasi asli?


---
## 9. Simpan hasil (Terorganisir per Tahap Analisis)

Seluruh berkas keluaran telah disusun ke dalam subfolder bernomor agar mudah ditelusuri (*traceability*) dan langsung siap dilampirkan sebagai suplementasi riset:

```text
results/
├── 01_qc/                      # Statistik SeqKit, qc_summary.tsv, & laporan NanoPlot
├── 02_classification/          # classification_summary.tsv, reports/ (.k2report), & per_read/ (.kraken.tsv)
├── 03_abundance/               # tables/ (matriks Genus & Spesies + LCA depth) & figures/ (barplot PNG)
├── 04_krona/                   # html/ (Krona interaktif) & tsv/ (input hierarki)
├── 05_diversity/               # alpha/ (rarefaction, Shannon/Simpson, boxplot) & beta/ (Bray-Curtis, PCoA, PERMANOVA)
├── 06_differential_abundance/  # Uji Mann-Whitney + BH-FDR (Genus) & ringkasan Spesies terarah
└── run_manifest.json           # Metadata jejak audit (timestamp, parameter, SHA-256 database, daftar sampel)
```

*(FASTQ mentah & terfilter berada di luar `results/` sehingga arsip ZIP tetap ringkas.)*


In [ ]:
# @title 9.1 Catat run_manifest.json, arsipkan hasil terorganisir, dan unduh ZIP
import json
from datetime import datetime, timezone

manifest_meta = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "PRJNA637202 (PreventADALL; Angell et al., 2020, PeerJ)",
    "database": {
        "name": "ONT EPI2ME NCBI Targeted Loci 16S/18S",
        "url": DB_URL,
        "expected_sha256": EXPECTED_SHA256,
    },
    "parameters": {
        "min_length": MIN_LENGTH,
        "max_length": MAX_LENGTH,
        "min_mean_q": MIN_MEAN_Q,
        "kraken_confidence": KRAKEN_CONFIDENCE,
        "rarefaction_depth_genus": int(RAREFACTION_DEPTH),
        "seed": SEED,
        "n_samples": int(len(samples)),
    },
    "samples": samples[["sample", "group", "run"]].to_dict(orient="records"),
}
manifest_path = RESULTS_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest_meta, indent=2), encoding="utf-8")

print("📂 Ringkasan struktur folder results/:")
for sub in sorted(p for p in RESULTS_DIR.rglob("*") if p.is_dir()):
    n_files = sum(1 for f in sub.iterdir() if f.is_file())
    if n_files > 0:
        print(f"   • {sub.relative_to(RESULTS_DIR)}/ ({n_files} file)")

archive_base = WORKDIR / "16s_workshop_results"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=WORKDIR, base_dir="results")
print(f"\n✅ Arsip siap: {archive} ({Path(archive).stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Bukan Colab; ambil file secara manual.")


## Checklist sebelum melaporkan hasil

- [ ] Accession, barcode, alias ENA, dan kelompok sampel dicatat.
- [ ] Versi alat (`kraken2`, `chopper`), database (`NCBI Targeted Loci 16S/18S`), parameter, dan seed dicatat.
- [ ] Retensi filtering diperiksa.
- [ ] Classified/unclassified dilaporkan.
- [ ] Kedalaman distandardisasi via rarefaction sebelum perbandingan diversity (Genus) & eksplorasi Spesies.
- [ ] Keterbatasan resolusi spesies pada gen 16S dan potensi bias primer dibahas.
- [ ] Multiple testing dikoreksi (Benjamini–Hochberg FDR) dan ukuran sampel dilaporkan.
- [ ] Hasil diperlakukan sebagai asosiasi eksploratif, bukan efek kausal atau diagnosis klinis.

**Sumber utama data:** ENA/NCBI BioProject PRJNA637202 (Angell et al., 2020). **Database workshop:** ONT EPI2ME NCBI Targeted Loci 16S/18S Kraken2 archive. Untuk laporan ilmiah, cantumkan sitasi resmi versi alat dan database yang digunakan.